In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("fra_perfumes.csv")

In [3]:
df.head()

,Name,Gender,Rating Value,Rating Count,Main Accords,Perfumers,Description,url
0,9am Afnanfor women,for women,3.73,174,"['citrus', 'musky', 'woody', 'aromatic', 'warm...",[],9ambyAfnanis a fragrance for women. Top notes ...,https://www.fragrantica.com/perfume/Afnan/9am-...
1,9am Dive Afnanfor women and men,for women and men,4.29,842,"['fruity', 'woody', 'green', 'warm spicy', 'ar...",[],9am DivebyAfnanis a Aromatic Aquatic fragrance...,https://www.fragrantica.com/perfume/Afnan/9am-...
2,9am pour Femme Afnanfor women,for women,4.00,68,"['fruity', 'musky', 'amber', 'citrus', 'powder...",[],9am pour FemmebyAfnanis a Amber fragrance for ...,https://www.fragrantica.com/perfume/Afnan/9am-...
3,9pm Afnanfor men,for men,4.50,"6,865","['vanilla', 'amber', 'warm spicy', 'cinnamon',...",[],9pmbyAfnanis a Amber Vanilla fragrance for men...,https://www.fragrantica.com/perfume/Afnan/9pm-...
4,9pm pour Femme Afnanfor women,for women,3.49,63,"['woody', 'aromatic', 'rose', 'fruity', 'powde...",[],9pm pour FemmebyAfnanis a Amber Floral fragran...,https://www.fragrantica.com/perfume/Afnan/9pm-...


In [4]:
df.shape

(70103, 8)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70103 entries, 0 to 70102
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Name          70100 non-null  object 
 1   Gender        70100 non-null  object 
 2   Rating Value  63922 non-null  float64
 3   Rating Count  63922 non-null  object 
 4   Main Accords  70103 non-null  object 
 5   Perfumers     70103 non-null  object 
 6   Description   70100 non-null  object 
 7   url           70103 non-null  object 
dtypes: float64(1), object(7)
memory usage: 4.3+ MB


In [6]:
df.isnull().sum()

Name               3
Gender             3
Rating Value    6181
Rating Count    6181
Main Accords       0
Perfumers          0
Description        3
url                0
dtype: int64

In [7]:
df.duplicated().sum()

np.int64(112)

# Information
### Rows: 70103
### Columns: 8
### Missing values: 
Name               3

Gender             3

Rating Value    6181

Rating Count    6181

Description        3
### Duplicate rows: 112

In [8]:
df = df.drop_duplicates()

In [9]:
df = df.dropna(subset=["Name","Gender","Description"])

In [10]:
df = df.dropna(subset=["Rating Value","Rating Count"])

In [11]:
df.isnull().sum()

Name            0
Gender          0
Rating Value    0
Rating Count    0
Main Accords    0
Perfumers       0
Description     0
url             0
dtype: int64

In [12]:
df[["Rating Value","Rating Count"]].info()

<class 'pandas.core.frame.DataFrame'>
Index: 63814 entries, 0 to 70102
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Rating Value  63814 non-null  float64
 1   Rating Count  63814 non-null  object 
dtypes: float64(1), object(1)
memory usage: 1.5+ MB


In [13]:
df["Rating Count"] = df["Rating Count"].str.replace(",", "")

In [14]:
df["Rating Count"] = pd.to_numeric(df["Rating Count"])

In [15]:
df[["Rating Value","Rating Count"]].info()

<class 'pandas.core.frame.DataFrame'>
Index: 63814 entries, 0 to 70102
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Rating Value  63814 non-null  float64
 1   Rating Count  63814 non-null  int64  
dtypes: float64(1), int64(1)
memory usage: 1.5 MB


In [16]:
df["Name"] = df["Name"].str.replace("for women and men","", regex=False)
df["Name"] = df["Name"].str.replace("for women","", regex=False)
df["Name"] = df["Name"].str.replace("for men","", regex=False)

In [17]:
df["Brand"] = df["url"].str.extract(r"perfume/([^/]+)/")

In [18]:
df["Brand"] = df["Brand"].str.replace("-", " ")

In [19]:
df["Perfume"] = df.apply(
    lambda x: x["Name"].replace(x["Brand"], "").strip(),
    axis=1
)

In [20]:
df[["Perfume","Brand","Gender"]].head(10)

,Perfume,Brand,Gender
0,9am,Afnan,for women
1,9am Dive,Afnan,for women and men
2,9am pour Femme,Afnan,for women
3,9pm,Afnan,for men
4,9pm pour Femme,Afnan,for women
5,Naseej Al Kiswah,Afnan,for women and men
6,Naseej Al Oud,Afnan,for women and men
7,Naseej Al Ward,Afnan,for women and men
8,Naseej Al Zafaran,Afnan,for women and men
10,Black Oudh,Al Haramain Perfumes,for women and men


In [21]:
df["Brand"].nunique()

1742

In [22]:
df.duplicated(subset=["Perfume","Brand"]).sum()

np.int64(805)

In [23]:
df = df.drop_duplicates(subset=["Perfume","Brand"])

In [24]:
df["Gender"].value_counts()

Gender
for women and men    26145
for women            25823
for men              11041
Name: count, dtype: int64

In [25]:
df["Gender"] = df["Gender"].replace({
    "for women": "Women",
    "for men": "Men",
    "for women and men": "Unisex"
})

In [26]:
df["Gender"].value_counts()

Gender
Unisex    26145
Women     25823
Men       11041
Name: count, dtype: int64

In [27]:
df['Main Accords'] = df['Main Accords'].str.replace(r"[\[\]']", "", regex=True)

In [28]:
df['Perfumers'] = df['Perfumers'].str.replace(r"[\[\]']", "", regex=True)

In [29]:
pd.set_option('future.no_silent_downcasting', True)

In [30]:
df['Main Accords'] = df['Main Accords'].replace(["", "nan"], np.nan)
df['Perfumers'] = df['Perfumers'].replace(["", "nan"], np.nan)

In [31]:
display(df[['Name', 'Main Accords', 'Perfumers']].head(10))

,Name,Main Accords,Perfumers
0,9am Afnan,"citrus, musky, woody, aromatic, warm spicy, la...",NaN
1,9am Dive Afnan,"fruity, woody, green, warm spicy, aromatic, ci...",NaN
2,9am pour Femme Afnan,"fruity, musky, amber, citrus, powdery, sweet, ...",NaN
3,9pm Afnan,"vanilla, amber, warm spicy, cinnamon, sweet, f...",NaN
4,9pm pour Femme Afnan,"woody, aromatic, rose, fruity, powdery, violet...",NaN
5,Naseej Al Kiswah Afnan,"woody, amber, leather, oud, patchouli, animali...",NaN
6,Naseej Al Oud Afnan,"oud, leather, woody, amber, soft spicy, animal...",NaN
7,Naseej Al Ward Afnan,"rose, oud, fruity, musky, sweet, floral, powde...",NaN
8,Naseej Al Zafaran Afnan,"warm spicy, metallic, leather, fresh spicy, wo...",NaN
10,Black Oudh Al Haramain Perfumes,"woody, powdery, musky, amber, patchouli, vanil...",NaN


In [32]:
df[['Main Accords','Perfumers']].head(10)

,Main Accords,Perfumers
0,"citrus, musky, woody, aromatic, warm spicy, la...",NaN
1,"fruity, woody, green, warm spicy, aromatic, ci...",NaN
2,"fruity, musky, amber, citrus, powdery, sweet, ...",NaN
3,"vanilla, amber, warm spicy, cinnamon, sweet, f...",NaN
4,"woody, aromatic, rose, fruity, powdery, violet...",NaN
5,"woody, amber, leather, oud, patchouli, animali...",NaN
6,"oud, leather, woody, amber, soft spicy, animal...",NaN
7,"rose, oud, fruity, musky, sweet, floral, powde...",NaN
8,"warm spicy, metallic, leather, fresh spicy, wo...",NaN
10,"woody, powdery, musky, amber, patchouli, vanil...",NaN


In [33]:
df.replace(["", "nan", "Unknown"], np.nan, inplace=True)

In [34]:
df.drop(columns=["Name"], inplace=True)

In [35]:
cols = ["Perfume", "Brand"] + [col for col in df.columns if col not in ["Perfume", "Brand"]]
df = df[cols]

In [36]:
df.head(3)

,Perfume,Brand,Gender,Rating Value,Rating Count,Main Accords,Perfumers,Description,url
0,9am,Afnan,Women,3.73,174,"citrus, musky, woody, aromatic, warm spicy, la...",NaN,9ambyAfnanis a fragrance for women. Top notes ...,https://www.fragrantica.com/perfume/Afnan/9am-...
1,9am Dive,Afnan,Unisex,4.29,842,"fruity, woody, green, warm spicy, aromatic, ci...",NaN,9am DivebyAfnanis a Aromatic Aquatic fragrance...,https://www.fragrantica.com/perfume/Afnan/9am-...
2,9am pour Femme,Afnan,Women,4.00,68,"fruity, musky, amber, citrus, powdery, sweet, ...",NaN,9am pour FemmebyAfnanis a Amber fragrance for ...,https://www.fragrantica.com/perfume/Afnan/9am-...


In [37]:
df.isnull().sum()

Perfume           118
Brand               0
Gender              0
Rating Value        0
Rating Count        0
Main Accords     1116
Perfumers       37719
Description         0
url                 0
dtype: int64

In [38]:
df.duplicated(subset=["Perfume","Brand"]).sum()

np.int64(0)

In [39]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 63009 entries, 0 to 70102
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Perfume       62891 non-null  object 
 1   Brand         63009 non-null  object 
 2   Gender        63009 non-null  object 
 3   Rating Value  63009 non-null  float64
 4   Rating Count  63009 non-null  int64  
 5   Main Accords  61893 non-null  object 
 6   Perfumers     25290 non-null  object 
 7   Description   63009 non-null  object 
 8   url           63009 non-null  object 
dtypes: float64(1), int64(1), object(7)
memory usage: 4.8+ MB


In [40]:
df[df["Rating Count"] < 0]

,Perfume,Brand,Gender,Rating Value,Rating Count,Main Accords,Perfumers,Description,url


In [41]:
df["Brand"].nunique()
df["Brand"].value_counts().head(10)

Brand
Avon                 1211
The Dua Brand        1100
Zara                  910
Victoria s Secret     734
Bath Body Works       555
O Boticario           532
Guerlain              514
Jequiti               495
Dzintars              440
Demeter Fragrance     424
Name: count, dtype: int64

In [42]:
df["url"].str.contains("fragrantica").mean()

np.float64(1.0)

In [43]:
df.dropna(subset=["Perfume"], inplace=True)
df.reset_index(drop=True, inplace=True)

In [44]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62891 entries, 0 to 62890
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Perfume       62891 non-null  object 
 1   Brand         62891 non-null  object 
 2   Gender        62891 non-null  object 
 3   Rating Value  62891 non-null  float64
 4   Rating Count  62891 non-null  int64  
 5   Main Accords  61776 non-null  object 
 6   Perfumers     25223 non-null  object 
 7   Description   62891 non-null  object 
 8   url           62891 non-null  object 
dtypes: float64(1), int64(1), object(7)
memory usage: 4.3+ MB


In [45]:
df.describe(include='all')

,Perfume,Brand,Gender,Rating Value,Rating Count,Main Accords,Perfumers,Description,url
count,62891,62891,62891,62891.000000,62891.000000,61776,25223,62891,62891
unique,57450,1738,3,NaN,NaN,57590,2096,62891,62891
top,Rose,Avon,Unisex,NaN,NaN,"oud, fresh spicy",Alberto Morillas,9ambyAfnanis a fragrance for women. Top notes ...,https://www.fragrantica.com/perfume/Afnan/9am-...
freq,42,1211,26137,NaN,NaN,132,350,1,1
mean,NaN,NaN,NaN,3.976482,224.472389,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,0.523477,932.899127,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,1.000000,1.000000,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,3.750000,6.000000,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,4.000000,26.000000,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,4.250000,114.000000,NaN,NaN,NaN,NaN


In [46]:
df.columns = df.columns.str.lower().str.strip().str.replace(' ', '_')

In [47]:
df.to_csv("perfume_clean_dataset.csv", index=False)

In [48]:
import pandas as pd
import numpy as np

In [49]:
df = pd.read_csv("fra_perfumes.csv")

In [50]:
df.columns

Index(['Name', 'Gender', 'Rating Value', 'Rating Count', 'Main Accords',
       'Perfumers', 'Description', 'url'],
      dtype='object')

In [51]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70103 entries, 0 to 70102
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Name          70100 non-null  object 
 1   Gender        70100 non-null  object 
 2   Rating Value  63922 non-null  float64
 3   Rating Count  63922 non-null  object 
 4   Main Accords  70103 non-null  object 
 5   Perfumers     70103 non-null  object 
 6   Description   70100 non-null  object 
 7   url           70103 non-null  object 
dtypes: float64(1), object(7)
memory usage: 4.3+ MB


In [52]:
df.isnull().sum()

Name               3
Gender             3
Rating Value    6181
Rating Count    6181
Main Accords       0
Perfumers          0
Description        3
url                0
dtype: int64

In [53]:
df.duplicated().sum()

np.int64(112)

# Information
### Rows: 70103
### Columns: 8
### Missing values: 
Name               3

Gender             3

Rating Value    6181

Rating Count    6181

Description        3
### Duplicate rows: 112

In [54]:
df = df.drop_duplicates()

In [55]:
df = df.dropna(subset=["Name","Gender","Description"])

In [56]:
df = df.dropna(subset=["Rating Value","Rating Count"])

In [57]:
df.shape

(63814, 8)

In [58]:
df.isnull().sum()

Name            0
Gender          0
Rating Value    0
Rating Count    0
Main Accords    0
Perfumers       0
Description     0
url             0
dtype: int64

In [59]:
df[["Rating Value","Rating Count"]].info()

<class 'pandas.core.frame.DataFrame'>
Index: 63814 entries, 0 to 70102
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Rating Value  63814 non-null  float64
 1   Rating Count  63814 non-null  object 
dtypes: float64(1), object(1)
memory usage: 1.5+ MB


In [60]:
df["Rating Count"] = df["Rating Count"].str.replace(",", "")

In [61]:
df["Rating Count"] = pd.to_numeric(df["Rating Count"])

In [62]:
df[["Rating Value","Rating Count"]].info()

<class 'pandas.core.frame.DataFrame'>
Index: 63814 entries, 0 to 70102
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Rating Value  63814 non-null  float64
 1   Rating Count  63814 non-null  int64  
dtypes: float64(1), int64(1)
memory usage: 1.5 MB


In [63]:
df["Name"] = df["Name"].str.replace("for women and men","", regex=False)
df["Name"] = df["Name"].str.replace("for women","", regex=False)
df["Name"] = df["Name"].str.replace("for men","", regex=False)

In [64]:
df["Brand"] = df["url"].str.extract(r"perfume/([^/]+)/")

In [65]:
df["Brand"] = df["Brand"].str.replace("-", " ")

In [66]:
df["Perfume"] = df.apply(
    lambda x: x["Name"].replace(x["Brand"], "").strip(),
    axis=1
)

In [67]:
df[["Perfume","Brand","Gender"]].head(10)

,Perfume,Brand,Gender
0,9am,Afnan,for women
1,9am Dive,Afnan,for women and men
2,9am pour Femme,Afnan,for women
3,9pm,Afnan,for men
4,9pm pour Femme,Afnan,for women
5,Naseej Al Kiswah,Afnan,for women and men
6,Naseej Al Oud,Afnan,for women and men
7,Naseej Al Ward,Afnan,for women and men
8,Naseej Al Zafaran,Afnan,for women and men
10,Black Oudh,Al Haramain Perfumes,for women and men


In [68]:
df["Brand"].nunique()

1742

In [69]:
df.duplicated(subset=["Perfume","Brand"]).sum()

np.int64(805)

In [70]:
df = df.drop_duplicates(subset=["Perfume","Brand"])

In [71]:
df["Gender"].value_counts()

Gender
for women and men    26145
for women            25823
for men              11041
Name: count, dtype: int64

In [72]:
df["Gender"] = df["Gender"].replace({
    "for women": "Women",
    "for men": "Men",
    "for women and men": "Unisex"
})

In [73]:
df["Gender"].value_counts()

Gender
Unisex    26145
Women     25823
Men       11041
Name: count, dtype: int64

In [74]:
df['Main Accords'] = df['Main Accords'].str.replace(r"[\[\]']", "", regex=True)

In [75]:
df['Perfumers'] = df['Perfumers'].str.replace(r"[\[\]']", "", regex=True)

In [76]:
pd.set_option('future.no_silent_downcasting', True)

In [77]:
df['Main Accords'] = df['Main Accords'].replace(["", "nan"], np.nan)
df['Perfumers'] = df['Perfumers'].replace(["", "nan"], np.nan)

In [78]:
display(df[['Name', 'Main Accords', 'Perfumers']].head(10))

,Name,Main Accords,Perfumers
0,9am Afnan,"citrus, musky, woody, aromatic, warm spicy, la...",NaN
1,9am Dive Afnan,"fruity, woody, green, warm spicy, aromatic, ci...",NaN
2,9am pour Femme Afnan,"fruity, musky, amber, citrus, powdery, sweet, ...",NaN
3,9pm Afnan,"vanilla, amber, warm spicy, cinnamon, sweet, f...",NaN
4,9pm pour Femme Afnan,"woody, aromatic, rose, fruity, powdery, violet...",NaN
5,Naseej Al Kiswah Afnan,"woody, amber, leather, oud, patchouli, animali...",NaN
6,Naseej Al Oud Afnan,"oud, leather, woody, amber, soft spicy, animal...",NaN
7,Naseej Al Ward Afnan,"rose, oud, fruity, musky, sweet, floral, powde...",NaN
8,Naseej Al Zafaran Afnan,"warm spicy, metallic, leather, fresh spicy, wo...",NaN
10,Black Oudh Al Haramain Perfumes,"woody, powdery, musky, amber, patchouli, vanil...",NaN


In [79]:
df[['Main Accords','Perfumers']].head(10)

,Main Accords,Perfumers
0,"citrus, musky, woody, aromatic, warm spicy, la...",NaN
1,"fruity, woody, green, warm spicy, aromatic, ci...",NaN
2,"fruity, musky, amber, citrus, powdery, sweet, ...",NaN
3,"vanilla, amber, warm spicy, cinnamon, sweet, f...",NaN
4,"woody, aromatic, rose, fruity, powdery, violet...",NaN
5,"woody, amber, leather, oud, patchouli, animali...",NaN
6,"oud, leather, woody, amber, soft spicy, animal...",NaN
7,"rose, oud, fruity, musky, sweet, floral, powde...",NaN
8,"warm spicy, metallic, leather, fresh spicy, wo...",NaN
10,"woody, powdery, musky, amber, patchouli, vanil...",NaN


In [80]:
df.replace(["", "nan", "Unknown"], np.nan, inplace=True)

In [81]:
df.drop(columns=["Name"], inplace=True)

In [82]:
cols = ["Perfume", "Brand"] + [col for col in df.columns if col not in ["Perfume", "Brand"]]
df = df[cols]

In [83]:
df.isnull().sum()

Perfume           118
Brand               0
Gender              0
Rating Value        0
Rating Count        0
Main Accords     1116
Perfumers       37719
Description         0
url                 0
dtype: int64

In [84]:
df.duplicated(subset=["Perfume","Brand"]).sum()

np.int64(0)

In [85]:
df[df["Rating Count"] < 0]

,Perfume,Brand,Gender,Rating Value,Rating Count,Main Accords,Perfumers,Description,url


In [86]:
df["Brand"].nunique()
df["Brand"].value_counts().head(10)

Brand
Avon                 1211
The Dua Brand        1100
Zara                  910
Victoria s Secret     734
Bath Body Works       555
O Boticario           532
Guerlain              514
Jequiti               495
Dzintars              440
Demeter Fragrance     424
Name: count, dtype: int64

In [87]:
df["url"].str.contains("fragrantica").mean()

np.float64(1.0)

In [88]:
df.dropna(subset=["Perfume"], inplace=True)
df.reset_index(drop=True, inplace=True)

In [89]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62891 entries, 0 to 62890
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Perfume       62891 non-null  object 
 1   Brand         62891 non-null  object 
 2   Gender        62891 non-null  object 
 3   Rating Value  62891 non-null  float64
 4   Rating Count  62891 non-null  int64  
 5   Main Accords  61776 non-null  object 
 6   Perfumers     25223 non-null  object 
 7   Description   62891 non-null  object 
 8   url           62891 non-null  object 
dtypes: float64(1), int64(1), object(7)
memory usage: 4.3+ MB


In [90]:
df.describe(include='all')

,Perfume,Brand,Gender,Rating Value,Rating Count,Main Accords,Perfumers,Description,url
count,62891,62891,62891,62891.000000,62891.000000,61776,25223,62891,62891
unique,57450,1738,3,NaN,NaN,57590,2096,62891,62891
top,Rose,Avon,Unisex,NaN,NaN,"oud, fresh spicy",Alberto Morillas,9ambyAfnanis a fragrance for women. Top notes ...,https://www.fragrantica.com/perfume/Afnan/9am-...
freq,42,1211,26137,NaN,NaN,132,350,1,1
mean,NaN,NaN,NaN,3.976482,224.472389,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,0.523477,932.899127,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,1.000000,1.000000,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,3.750000,6.000000,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,4.000000,26.000000,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,4.250000,114.000000,NaN,NaN,NaN,NaN


In [91]:
df.head(3)

,Perfume,Brand,Gender,Rating Value,Rating Count,Main Accords,Perfumers,Description,url
0,9am,Afnan,Women,3.73,174,"citrus, musky, woody, aromatic, warm spicy, la...",NaN,9ambyAfnanis a fragrance for women. Top notes ...,https://www.fragrantica.com/perfume/Afnan/9am-...
1,9am Dive,Afnan,Unisex,4.29,842,"fruity, woody, green, warm spicy, aromatic, ci...",NaN,9am DivebyAfnanis a Aromatic Aquatic fragrance...,https://www.fragrantica.com/perfume/Afnan/9am-...
2,9am pour Femme,Afnan,Women,4.00,68,"fruity, musky, amber, citrus, powdery, sweet, ...",NaN,9am pour FemmebyAfnanis a Amber fragrance for ...,https://www.fragrantica.com/perfume/Afnan/9am-...


In [92]:
df.columns = df.columns.str.lower().str.strip().str.replace(" ", "_")

In [93]:
import re

mask = df.apply(lambda col: col.astype(str).str.contains(r"[^\x00-\x7F]", na=False))

df[mask.any(axis=1)]

,perfume,brand,gender,rating_value,rating_count,main_accords,perfumers,description,url
17,Étoiles Silver,Al Haramain Perfumes,Men,3.65,20,"fresh spicy, woody, citrus, aromatic, aquatic,...",NaN,Étoiles SilverbyAl Haramain Perfumesis a Woody...,https://www.fragrantica.com/perfume/Al-Haramai...
86,50 Years Oudh Ma’al Attar,Al Haramain Perfumes,Unisex,2.00,3,"woody, citrus, powdery, floral, green, fresh, ...",NaN,50 Years Oudh Ma’al AttarbyAl Haramain Perfume...,https://www.fragrantica.com/perfume/Al-Haramai...
122,Détour Rouge,Al Haramain Perfumes,Unisex,3.94,99,"aromatic, powdery, musky, warm spicy, sweet, w...",NaN,Détour RougebyAl Haramain Perfumesis a Citrus ...,https://www.fragrantica.com/perfume/Al-Haramai...
404,God Is A Woman,Ariana Grande,Women,3.89,2470,"fruity, sweet, powdery, vanilla, aquatic, musk...",Jérôme Epinette,God Is A WomanbyAriana Grandeis a Amber Floral...,https://www.fragrantica.com/perfume/Ariana-Gra...
411,God Is A Woman Body Mist,Ariana Grande,Women,4.00,9,"vanilla, fruity, sweet, powdery, floral, musky...",Jérôme Epinette,God Is A Woman Body MistbyAriana Grandeis a Am...,https://www.fragrantica.com/perfume/Ariana-Gra...
...,...,...,...,...,...,...,...,...,...
62786,Petit Amande,Adopt Parfums,Women,3.93,15,"almond, sweet, nutty, fruity, citrus",Dominique Monlun,Petit AmandebyAdopt Parfumsis a Citrus Gourman...,https://www.fragrantica.com/perfume/Adopt-Parf...
62798,Pêche Mandarine,Adopt Parfums,Women,3.55,11,"fruity, musky, white floral, powdery, citrus, ...",NaN,Pêche MandarinebyAdopt Parfumsis a Floral Frui...,https://www.fragrantica.com/perfume/Adopt-Parf...
62810,Détour Noir,Al Haramain Perfumes,Unisex,4.41,2698,"vanilla, woody, powdery, warm spicy, fresh spi...",NaN,Détour NoirbyAl Haramain Perfumesis a Amber Sp...,https://www.fragrantica.com/perfume/Al-Haramai...
62820,Pêche Ananas Cèdre,Adopt Parfums,Women,4.12,8,"fruity, sweet, rose, woody, tropical, powdery",NaN,Pêche Ananas CèdrebyAdopt Parfumsis a Floral F...,https://www.fragrantica.com/perfume/Adopt-Parf...


In [94]:
df[df["perfume"].str.contains("Ã|â|€", na=False)]

,perfume,brand,gender,rating_value,rating_count,main_accords,perfumers,description,url
1128,Aquavibe Mix Alzafema + Sândalo,Avon,Unisex,4.18,17,"woody, lavender, powdery, aromatic, warm spicy...",NaN,Aquavibe Mix Alzafema + SândalobyAvonis a Flor...,https://www.fragrantica.com/perfume/Avon/Aquav...
1289,Breeze Sândalo Fresh,Avon,Women,4.33,3,"citrus, woody, white floral, aromatic, fresh, ...",Francisca Perez Areias,Breeze Sândalo FreshbyAvonis a Amber Woody fra...,https://www.fragrantica.com/perfume/Avon/Breez...
1328,Sândalo Fresh,Avon,Women,4.14,58,"woody, citrus, aromatic, fresh, fruity, powder...",NaN,Sândalo FreshbyAvonis a Floral Fruity fragranc...,https://www.fragrantica.com/perfume/Avon/Sanda...
1376,Encanto Elegância,Avon,Women,3.67,3,"fruity, green, nutty, sweet, balsamic, warm spicy",NaN,Encanto ElegânciabyAvonis a Floral Fruity frag...,https://www.fragrantica.com/perfume/Avon/Encan...
1377,Encanto Elegância Castanha,Avon,Women,4.60,5,"floral, nutty, balsamic",NaN,Encanto Elegância CastanhabyAvonis a Floral fr...,https://www.fragrantica.com/perfume/Avon/Encan...
...,...,...,...,...,...,...,...,...,...
60700,Intense Gold Câline,Caline,Men,4.19,31,"amber, cinnamon, warm spicy, fresh spicy, vani...",NaN,Intense GoldbyCâlineis a Aromatic Spicy fragra...,https://www.fragrantica.com/perfume/Caline/Int...
60701,Jour En Rose Câline,Caline,Women,3.84,19,"sweet, white floral, vanilla, rose, fruity, fl...",NaN,Jour En RosebyCâlineis a Floral fragrance for ...,https://www.fragrantica.com/perfume/Caline/Jou...
60702,Mon Amour Câline,Caline,Women,4.26,35,"white floral, amber, powdery, woody, musky, fl...",NaN,Mon AmourbyCâlineis a Amber fragrance for wome...,https://www.fragrantica.com/perfume/Caline/Mon...
60703,Rouge Intense Câline,Caline,Women,4.10,31,"vanilla, powdery, almond, sweet, fruity, woody...",NaN,Rouge IntensebyCâlineis a Floral Fruity fragra...,https://www.fragrantica.com/perfume/Caline/Rou...


In [95]:
import unicodedata

def normalize_text(x):
    if isinstance(x, str):
        x = unicodedata.normalize("NFKC", x)
        return x.strip()
    return x

cols = ["perfume","brand","description"]

for c in cols:
    df[c] = df[c].apply(normalize_text)

In [96]:
import re

def remove_control_chars(text):
    if isinstance(text,str):
        return re.sub(r"[\x00-\x1F\x7F]", "", text)
    return text

for c in ["perfume","brand","description"]:
    df[c] = df[c].apply(remove_control_chars)

In [97]:
def fix_encoding(text):
    if isinstance(text,str):
        try:
            return text.encode("latin1").decode("utf8")
        except:
            return text
    return text

for c in ["perfume","brand","description"]:
    df[c] = df[c].apply(fix_encoding)

In [98]:
df["perfume"] = df["perfume"].str.replace(r"\s+"," ",regex=True)
df["brand"] = df["brand"].str.replace(r"\s+"," ",regex=True)
df["description"] = df["description"].str.replace(r"\s+"," ",regex=True)

In [99]:
df.sample(20)
df[df["perfume"].str.contains(r"[^\x00-\x7F]", na=False)].head()

,perfume,brand,gender,rating_value,rating_count,main_accords,perfumers,description,url
17,Étoiles Silver,Al Haramain Perfumes,Men,3.65,20,"fresh spicy, woody, citrus, aromatic, aquatic,...",NaN,Étoiles SilverbyAl Haramain Perfumesis a Woody...,https://www.fragrantica.com/perfume/Al-Haramai...
86,50 Years Oudh Ma’al Attar,Al Haramain Perfumes,Unisex,2.00,3,"woody, citrus, powdery, floral, green, fresh, ...",NaN,50 Years Oudh Ma’al AttarbyAl Haramain Perfume...,https://www.fragrantica.com/perfume/Al-Haramai...
122,Détour Rouge,Al Haramain Perfumes,Unisex,3.94,99,"aromatic, powdery, musky, warm spicy, sweet, w...",NaN,Détour RougebyAl Haramain Perfumesis a Citrus ...,https://www.fragrantica.com/perfume/Al-Haramai...
678,Amor & Vida Bebê,Avon,Unisex,4.75,4,"lavender, musky, powdery, aromatic, fresh spic...",NaN,Amor & Vida BebêbyAvonis a Aromatic Green frag...,https://www.fragrantica.com/perfume/Avon/Amor-...
679,Amor & Vida Mamãe,Avon,Women,3.00,1,"citrus, floral, musky, fresh, powdery",NaN,Amor & Vida MamãebyAvonis a Floral fragrance f...,https://www.fragrantica.com/perfume/Avon/Amor-...


In [100]:
df.to_csv("perfumes_cleaned.csv", index=False, encoding="utf-8-sig")

In [101]:
df.columns = df.columns.str.lower().str.replace(" ", "_")

In [102]:
df_accords = df.copy()

df_accords["main_accords"] = df_accords["main_accords"].str.split(",")

df_accords = df_accords.explode("main_accords")

df_accords["main_accords"] = df_accords["main_accords"].str.strip()

In [103]:
print(df["perfume"].nunique())
print(df["brand"].nunique())

57438
1738


In [104]:
df["gender"].value_counts()

gender
Unisex    26137
Women     25731
Men       11023
Name: count, dtype: int64

In [105]:
df["gender"].value_counts(normalize=True)

gender
Unisex    0.415592
Women     0.409136
Men       0.175272
Name: proportion, dtype: float64

In [106]:
df["brand"].value_counts().head(15)

brand
Avon                    1211
The Dua Brand           1100
Zara                     910
Victoria s Secret        734
Bath Body Works          555
O Boticario              532
Guerlain                 514
Jequiti                  495
Dzintars                 439
Demeter Fragrance        424
Natura                   416
Oriflame                 394
Al Haramain Perfumes     299
DSH Perfumes             297
Ajmal                    295
Name: count, dtype: int64

In [107]:
df["rating_value"].describe()

count    62891.000000
mean         3.976482
std          0.523477
min          1.000000
25%          3.750000
50%          4.000000
75%          4.250000
max          5.000000
Name: rating_value, dtype: float64

In [108]:
df.sort_values("rating_count", ascending=False).head(10)

,perfume,brand,gender,rating_value,rating_count,main_accords,perfumers,description,url
11007,Alien,Mugler,Women,4.00,29858,"white floral, amber, floral","Dominique Ropion, Laurent Bruyere",AlienbyMugleris a Amber Woody fragrance for wo...,https://www.fragrantica.com/perfume/Mugler/Ali...
11040,Angel,Mugler,Women,3.57,29722,"sweet, patchouli, fruity, warm spicy, caramel,...","Olivier Cresp, Yves de Chirin",AngelbyMugleris a Amber Vanilla fragrance for ...,https://www.fragrantica.com/perfume/Mugler/Ang...
5383,Light Blue Dolce&Gabbana,Dolce Gabbana,Women,3.84,29708,"citrus, woody, fresh, fruity, aromatic, musky,...",Olivier Cresp,Light BluebyDolce&Gabbanais a Floral Fruity fr...,https://www.fragrantica.com/perfume/Dolce-Gabb...
3795,Coco Mademoiselle,Chanel,Women,4.12,29283,"citrus, woody, patchouli, sweet, white floral,...",Jacques Polge,Coco MademoisellebyChanelis a Amber Floral fra...,https://www.fragrantica.com/perfume/Chanel/Coc...
9139,La Vie Est Belle Lancôme,Lancome,Women,3.64,28982,"sweet, vanilla, fruity, patchouli, woody, whit...",NaN,La Vie Est BellebyLancômeis a Floral Fruity Go...,https://www.fragrantica.com/perfume/Lancome/La...
14138,Black Orchid,Tom Ford,Women,3.91,26053,"warm spicy, earthy, woody, sweet, amber, patch...","David Apel, Pierre Negrin",Black OrchidbyTom Fordis a Amber Floral fragra...,https://www.fragrantica.com/perfume/Tom-Ford/B...
15962,Black Opium,Yves Saint Laurent,Women,3.93,25669,"vanilla, coffee, sweet, warm spicy, white flor...",NaN,Black OpiumbyYves Saint Laurentis a Amber Vani...,https://www.fragrantica.com/perfume/Yves-Saint...
5232,Hypnotic Poison,Dior,Women,4.09,25173,"vanilla, sweet, almond, fruity, nutty, powdery...","Annick Menardo, Christian Dussoulier",Hypnotic PoisonbyDioris a Amber Vanilla fragra...,https://www.fragrantica.com/perfume/Dior/Hypno...
5092,J'adore,Dior,Women,3.78,25013,"white floral, floral, fruity, sweet, fresh, aq...",Calice Becker,J'adorebyDioris a Floral Fruity fragrance for ...,https://www.fragrantica.com/perfume/Dior/J-ado...
5256,Sauvage,Dior,Men,3.88,23727,"fresh spicy, amber, citrus, aromatic, musky, w...",François Demachy,SauvagebyDioris a Aromatic Fougere fragrance f...,https://www.fragrantica.com/perfume/Dior/Sauva...


In [109]:
df["popularity_score"] = df["rating_value"] * np.log1p(df["rating_count"])

In [110]:
df.sort_values("popularity_score", ascending=False).head(10)

,perfume,brand,gender,rating_value,rating_count,main_accords,perfumers,description,url,popularity_score
5051,Homme Intense 2011,Dior,Men,4.50,18272,"iris, woody, powdery, earthy, violet, aromatic...",François Demachy,Dior Homme Intense 2011byDioris a Woody Floral...,https://www.fragrantica.com/perfume/Dior/Dior-...,44.159309
16041,La Nuit de l'Homme,Yves Saint Laurent,Men,4.44,19533,"aromatic, warm spicy, lavender, woody, fresh s...",NaN,La Nuit de l'HommebyYves Saint Laurentis a Woo...,https://www.fragrantica.com/perfume/Yves-Saint...,43.866808
8241,Le Male Le Parfum,Jean Paul Gaultier,Men,4.60,13406,"warm spicy, vanilla, lavender, aromatic, powde...","Natalie Gracia-Cetto, Quentin Bisch",Le Male Le ParfumbyJean Paul Gaultieris a Ambe...,https://www.fragrantica.com/perfume/Jean-Paul-...,43.716248
4343,Aventus,Creed,Men,4.34,19581,"fruity, sweet, leather, woody, smoky, citrus, ...","Erwin Creed, Jean-Christophe Hérault",AventusbyCreedis a Chypre Fruity fragrance for...,https://www.fragrantica.com/perfume/Creed/Aven...,42.889469
7811,Terre d'Hermès Hermès,Hermes,Men,4.28,21347,"citrus, woody, fresh spicy, aromatic, earthy, ...",Jean-Claude Ellena,Terre d'HermèsbyHermèsis a Woody Spicy fragran...,https://www.fragrantica.com/perfume/Hermes/Ter...,42.666093
3795,Coco Mademoiselle,Chanel,Women,4.12,29283,"citrus, woody, patchouli, sweet, white floral,...",Jacques Polge,Coco MademoisellebyChanelis a Amber Floral fra...,https://www.fragrantica.com/perfume/Chanel/Coc...,42.373362
14202,Tobacco Vanille,Tom Ford,Unisex,4.23,22332,"vanilla, sweet, tobacco, warm spicy, fruity, w...",Olivier Gillotin,Tobacco VanillebyTom Fordis a Amber Spicy frag...,https://www.fragrantica.com/perfume/Tom-Ford/T...,42.358461
8235,Le Male Elixir,Jean Paul Gaultier,Men,4.56,10388,"vanilla, sweet, honey, amber, aromatic, lavend...",Quentin Bisch,Le Male ElixirbyJean Paul Gaultieris a Amber F...,https://www.fragrantica.com/perfume/Jean-Paul-...,42.173173
16189,Y Eau de Parfum,Yves Saint Laurent,Men,4.38,15024,"aromatic, woody, fresh spicy, warm spicy, ambe...",Dominique Ropion,Y Eau de ParfumbyYves Saint Laurentis a Aromat...,https://www.fragrantica.com/perfume/Yves-Saint...,42.124522
5998,Acqua di Giò Profumo,Giorgio Armani,Men,4.41,14072,"aromatic, marine, fresh spicy, amber, woody, s...",Alberto Morillas,Acqua di Giò ProfumobyGiorgio Armaniis a Aroma...,https://www.fragrantica.com/perfume/Giorgio-Ar...,42.124379


In [111]:
df["perfume_id"] = range(1, len(df) + 1)

In [112]:
perfumes = df[
[
"perfume_id",
"perfume",
"brand",
"gender",
"rating_value",
"rating_count"
]
]

In [113]:
accords = df[["perfume_id", "main_accords"]].copy()

accords["main_accords"] = accords["main_accords"].str.split(",")

accords = accords.explode("main_accords")

accords["main_accords"] = accords["main_accords"].str.strip()

accords.rename(columns={"main_accords":"accord"}, inplace=True)

In [114]:
perfumes.to_csv("perfumes.csv", index=False)

In [115]:
accords.to_csv("perfume_accords.csv", index=False)